In [ ]:
import chess
import chess.pgn
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
from pathlib import Path
from tqdm.notebook import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
class AlphaZeroNet(nn.Module):
    def __init__(self, input_channels=12, board_size=8, num_actions=4288):
        super().__init__()
        self.conv1 = nn.Conv2d(input_channels, 64, 3, padding=1)
        self.conv2 = nn.Conv2d(64, 128, 3, padding=1)
        self.conv3 = nn.Conv2d(128, 128, 3, padding=1)

        self.fc_policy = nn.Linear(128 * board_size * board_size, num_actions)
        self.fc_value = nn.Linear(128 * board_size * board_size, 1)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = x.view(x.size(0), -1)

        policy = self.fc_policy(x)
        value = torch.tanh(self.fc_value(x))
        return policy, value

model = AlphaZeroNet().to(device)


In [ ]:
def encode_board(board: chess.Board):
    """Convert board to tensor (12x8x8)"""
    planes = np.zeros((12, 8, 8), dtype=np.float32)
    for sq, piece in board.piece_map().items():
        row = 7 - (sq // 8)
        col = sq % 8
        channel_offset = 6 if piece.color == chess.WHITE else 0
        planes[channel_offset + (piece.piece_type - 1), row, col] = 1.0
    return torch.tensor(planes, device=device)


In [ ]:
uci_to_idx = {}
idx_to_uci = []

def build_move_index():
    global uci_to_idx, idx_to_uci
    all_moves = []
    for a in range(64):
        for b in range(64):
            if a != b:
                all_moves.append(chess.Move(a, b))
    promos = [chess.QUEEN, chess.ROOK, chess.BISHOP, chess.KNIGHT]
    for a in range(64):
        rank = a // 8
        if rank == 6:
            for to_sq in range(56,64):
                for p in promos:
                    all_moves.append(chess.Move(a,to_sq,p))
    all_moves = sorted(all_moves,key=lambda m: m.uci())
    idx_to_uci = [m.uci() for m in all_moves]
    uci_to_idx = {m.uci():i for i,m in enumerate(all_moves)}
    print(f"[INFO] Move index built with {len(idx_to_uci)} actions.")

build_move_index()

def move_to_index(move: chess.Move):
    return uci_to_idx.get(move.uci(),0)


[INFO] Move index built with 4288 actions.


In [ ]:
class Node:
    def __init__(self,parent=None,prior=0.0):
        self.parent = parent
        self.children = {}
        self.N = 0
        self.W = 0
        self.Q = 0
        self.P = prior

def mcts(board, simulations=25, c_puct=1.0):
    root = Node()
    for _ in range(simulations):
        node = root
        sim_board = board.copy()

        # Selection
        while node.children:
            move, node = max(
                node.children.items(),
                key=lambda kv: kv[1].Q + c_puct * kv[1].P * (np.sqrt(node.N + 1e-8)/(1+kv[1].N))
            )
            sim_board.push(move)

        # Evaluate
        state = encode_board(sim_board).unsqueeze(0)
        with torch.no_grad():
            log_probs, value = model(state)
        probs = log_probs.exp().cpu().numpy()[0]

        # Mask legal moves
        legal_priors = {}
        s = 0
        for mv in sim_board.legal_moves:
            legal_priors[mv] = probs[move_to_index(mv)]
            s += probs[move_to_index(mv)]
        for mv in legal_priors:  # normalize
            legal_priors[mv] /= s + 1e-8

        # Expand
        for mv,p in legal_priors.items():
            node.children[mv] = Node(parent=node,prior=p)

        # Backprop
        v = float(value.item())
        cur = node
        while cur is not None:
            cur.N += 1
            cur.W += v
            cur.Q = cur.W / cur.N
            v = -v
            cur = cur.parent

    # Choose action
    moves, nodes = zip(*root.children.items())
    visits = [n.N for n in nodes]
    best_move = moves[int(np.argmax(visits))]
    return best_move


In [ ]:
def self_play_episode():
    board = chess.Board()
    states, pis, rewards = [], [], []

    while not board.is_game_over():
        move = mcts(board, simulations=25)
        # Collect state
        states.append(encode_board(board).cpu().numpy())
        # Collect policy target
        pi = np.zeros(len(idx_to_uci),dtype=np.float32)
        for mv in board.legal_moves:
            if mv == move:
                pi[move_to_index(mv)] = 1.0
        pis.append(pi)

        board.push(move)

    # Reward: +1 win, -1 loss, 0 draw
    result = board.result()
    if result=="1-0": r=1
    elif result=="0-1": r=-1
    else: r=0
    rewards = [r]*len(states)

    return np.array(states), np.array(pis), np.array(rewards)

In [18]:
from tqdm.notebook import tqdm

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
epochs = 10  # increase for more training

for ep in tqdm(range(epochs)):
    states, pis, rewards = self_play_episode()

    states = torch.tensor(states,device=device,dtype=torch.float32)
    pis = torch.tensor(pis,device=device,dtype=torch.float32)
    rewards = torch.tensor(rewards,device=device,dtype=torch.float32).unsqueeze(1)

    optimizer.zero_grad()
    log_probs, values = model(states)
    loss_p = -torch.sum(pis*F.log_softmax(log_probs,dim=1))/len(states)
    loss_v = F.mse_loss(values, rewards)
    loss = loss_p + loss_v
    loss.backward()
    optimizer.step()

    print(f"Episode {ep+1} loss: {loss.item():.4f}")


ImportError: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html

In [ ]:
torch.save(model.state_dict(), Path.cwd()/"trained_model.pt")
print("[INFO] Model saved as trained_model.pt")
